In [1]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True Tesla T4


In [2]:
!pip install -q -U transformers accelerate bitsandbytes datasets qwen-vl-utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 18.5 MB/s eta 0:00:00


In [3]:
from datasets import load_dataset
ds = load_dataset("lmms-lab/HallusionBench")
print(ds)

README.md:   0%|          | 0.00/2.66k [00:00<?, ?B/s]

data/image-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  147MB            

data/image-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/non_image-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 22.9kB            

data/non_image-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating image split:   0%|          | 0/951 [00:00<?, ? examples/s]

Generating non_image split:   0%|          | 0/178 [00:00<?, ? examples/s]

DatasetDict({
    image: Dataset({
        features: ['category', 'subcategory', 'visual_input', 'set_id', 'figure_id', 'sample_note', 'question_id', 'question', 'gt_answer_details', 'gt_answer', 'filename', 'image'],
        num_rows: 951
    })
    non_image: Dataset({
        features: ['category', 'subcategory', 'visual_input', 'set_id', 'figure_id', 'sample_note', 'question_id', 'question', 'gt_answer_details', 'gt_answer', 'filename', 'image'],
        num_rows: 178
    })
})


In [4]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from qwen_vl_utils import process_vision_info
import torch

model_id = "Qwen/Qwen2.5-VL-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, # true because less VRAM, running on colab
    bnb_4bit_compute_dtype=torch.float16,
)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)
processor = AutoProcessor.from_pretrained(model_id)

config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/57.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

...

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/5.70k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [5]:
example = ds["image"][0]

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": example["image"]},
            {"type": "text", "text": f"{example['question']} Please answer with only Yes or No."},
        ],
    }
]

text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
image_inputs, video_inputs = process_vision_info(messages)

inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
).to(model.device)

output = model.generate(**inputs, max_new_tokens=20)
generated_ids = output[:, inputs.input_ids.shape[1]:]  # trim off the echoed prompt tokens
response = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

print("Question:", example["question"])
print("Ground truth:", example["gt_answer"])
print("Model response:", response)

Question: Is China, Hongkong SAR, the leading importing country of gold, silverware, and jewelry with the highest import value in 2018?
Ground truth: 0
Model response: No.


In [6]:
def parse_yes_no(response_text):
    answer = response_text.strip().lower()
    if answer.startswith("yes"):
        return "1"
    elif answer.startswith("no"):
        return "0"
    else:
        return "unclear"

In [7]:
print(parse_yes_no("No."))  #parsing checkpoint

0


In [8]:
processor = AutoProcessor.from_pretrained(
    model_id,
    min_pixels=256*28*28,
    max_pixels=768*28*28,   # cap resolution — lower this further (e.g. 512*28*28) if OOM persists
)

In [9]:
import json, os

results_path = "/content/major-project/results/qwen2.5vl_results.json"
os.makedirs(os.path.dirname(results_path), exist_ok=True)

if os.path.exists(results_path):
    with open(results_path) as f:
        results = json.load(f)
else:
    results = []

done_ids = {(r["question_id"], r["split"]) for r in results}

def run_split(split_name):
    for i, example in enumerate(ds[split_name]):
        if (example["question_id"], split_name) in done_ids:
            continue

        content = []
        if example["image"] is not None:
            content.append({"type": "image", "image": example["image"]})
        content.append({"type": "text", "text": f"{example['question']} Please answer with only Yes or No."})

        messages = [{"role": "user", "content": content}]

        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)

        inputs = processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        ).to(model.device)

        output = model.generate(**inputs, max_new_tokens=20)
        generated_ids = output[:, inputs.input_ids.shape[1]:]
        raw_response = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

        pred = parse_yes_no(raw_response)

        results.append({
            "question_id": example["question_id"],
            "split": split_name,
            "category": example["category"],
            "subcategory": example["subcategory"],
            "figure_id": example["figure_id"],
            "set_id": example["set_id"],
            "question": example["question"],
            "gt_answer": example["gt_answer"],
            "raw_response": raw_response,
            "model_prediction": pred,
        })

        if (i + 1) % 50 == 0:
            with open(results_path, "w") as f:
                json.dump(results, f, indent=2)
            print(f"[{split_name}] Saved progress: {i+1}/{len(ds[split_name])}")

    with open(results_path, "w") as f:
        json.dump(results, f, indent=2)

run_split("image")
run_split("non_image")

print("Done:", len(results), "total questions processed")

/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:957: UserWarning: inner dimension (3420) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(


[image] Saved progress: 50/951
[image] Saved progress: 100/951
[image] Saved progress: 150/951
[image] Saved progress: 200/951
[image] Saved progress: 250/951
[image] Saved progress: 300/951
[image] Saved progress: 350/951
[image] Saved progress: 400/951
[image] Saved progress: 450/951
[image] Saved progress: 500/951


/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


[image] Saved progress: 550/951
[image] Saved progress: 600/951
[image] Saved progress: 650/951
[image] Saved progress: 700/951
[image] Saved progress: 750/951
[image] Saved progress: 800/951
[image] Saved progress: 850/951
[image] Saved progress: 900/951
[image] Saved progress: 950/951
[non_image] Saved progress: 50/178
[non_image] Saved progress: 100/178
[non_image] Saved progress: 150/178
Done: 1129 total questions processed


In [10]:
import time, gc

start = time.time()
for example in ds["image"].select(range(20)):
    content = [{"type": "image", "image": example["image"]},
               {"type": "text", "text": f"{example['question']} Please answer with only Yes or No."}]
    messages = [{"role": "user", "content": content}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt").to(model.device)
    output = model.generate(**inputs, max_new_tokens=20)

    del inputs, output
    gc.collect()
    torch.cuda.empty_cache()

elapsed = time.time() - start
print(f"Time for 20: {elapsed:.1f}s | Estimated full 1129: {elapsed * 1129/20/60:.1f} min")

Time for 20: 54.0s | Estimated full 1129: 50.8 min


In [13]:
import json
from collections import defaultdict

with open("/content/major-project/results/qwen2.5vl_results.json") as f:
    qwen_results = json.load(f)

def compute_metrics(results):
    total = len(results)
    if total == 0:
        return None
    correct = sum(1 for r in results if r["model_prediction"] == str(r["gt_answer"]))
    aAcc = correct / total * 100

    # qAcc: group by category + subcategory + set_id
    sets = defaultdict(list)
    for r in results:
        key = (r["category"], r["subcategory"], r["set_id"])
        sets[key].append(r)
    correct_sets = sum(
        1 for members in sets.values()
        if all(m["model_prediction"] == str(m["gt_answer"]) for m in members)
    )
    qAcc = correct_sets / len(sets) * 100

    # fAcc: group by category + subcategory + figure_id
    figures = defaultdict(list)
    for r in results:
        key = (r["category"], r["subcategory"], r["figure_id"])
        figures[key].append(r)
    correct_figures = sum(
        1 for members in figures.values()
        if all(m["model_prediction"] == str(m["gt_answer"]) for m in members)
    )
    fAcc = correct_figures / len(figures) * 100

    return {
        "aAcc": round(aAcc, 2),
        "qAcc": round(qAcc, 2),
        "fAcc": round(fAcc, 2),
        "n_questions": total,
        "n_sets": len(sets),
        "n_figures": len(figures),
    }


print("=== Overall ===")
print(compute_metrics(qwen_results))

by_cat = defaultdict(list)
for r in qwen_results:
    by_cat[r["category"]].append(r)
print("\n=== By Category ===")
for cat, items in by_cat.items():
    print(cat, compute_metrics(items))

by_subcat = defaultdict(list)
for r in qwen_results:
    by_subcat[(r["category"], r["subcategory"])].append(r)
print("\n=== By Category + Subcategory ===")
for key, items in sorted(by_subcat.items()):
    print(key, compute_metrics(items))

unclear_count = sum(1 for r in qwen_results if r["model_prediction"] == "unclear")
print(f"\nUnclear responses: {unclear_count}/{len(qwen_results)}")

=== Overall ===
{'aAcc': 69.8, 'qAcc': 15.57, 'fAcc': 3.85, 'n_questions': 1129, 'n_sets': 167, 'n_figures': 26}

=== By Category ===
VS {'aAcc': 72.86, 'qAcc': 10.53, 'fAcc': 0.0, 'n_questions': 538, 'n_sets': 57, 'n_figures': 13}
VD {'aAcc': 67.01, 'qAcc': 18.18, 'fAcc': 7.69, 'n_questions': 591, 'n_sets': 110, 'n_figures': 13}

=== By Category + Subcategory ===
('VD', 'figure') {'aAcc': 76.25, 'qAcc': 25.0, 'fAcc': 33.33, 'n_questions': 80, 'n_sets': 20, 'n_figures': 3}
('VD', 'illusion') {'aAcc': 62.5, 'qAcc': 9.68, 'fAcc': 0.0, 'n_questions': 144, 'n_sets': 31, 'n_figures': 2}
('VD', 'math') {'aAcc': 67.59, 'qAcc': 0.0, 'fAcc': 0.0, 'n_questions': 108, 'n_sets': 18, 'n_figures': 2}
('VD', 'ocr') {'aAcc': 84.27, 'qAcc': 52.38, 'fAcc': 0.0, 'n_questions': 89, 'n_sets': 21, 'n_figures': 3}
('VD', 'video') {'aAcc': 57.06, 'qAcc': 5.0, 'fAcc': 0.0, 'n_questions': 170, 'n_sets': 20, 'n_figures': 3}
('VS', 'chart') {'aAcc': 78.64, 'qAcc': 22.73, 'fAcc': 0.0, 'n_questions': 206, 'n_sets':

In [14]:
metrics_summary = {
    "model": "qwen2.5-vl-7b-instruct",
    "overall": compute_metrics(qwen_results),
    "by_category": {cat: compute_metrics(items) for cat, items in by_cat.items()},
    "by_category_subcategory": {f"{k[0]}_{k[1]}": compute_metrics(items) for k, items in by_subcat.items()},
}

with open("/content/major-project/results/qwen2.5vl_metrics.json", "w") as f:
    json.dump(metrics_summary, f, indent=2)